# Harmonized 72-Class Leaf Disease Model Comparison

This notebook rebuilds the canonical 70/15/15 benchmark from the attached Kaggle dataset, then trains the supported mobile classifiers and compares full INT8 TFLite quality, model size, and single-thread CPU latency. CPU latency is a proxy; validate finalists on the target Android device.

Attach the multi-crop leaf disease dataset in Kaggle Notebook settings and enable a GPU accelerator.
Set `COPY_DATASET_TO_WORKING = True` in the setup cell to optionally copy the discovered raw/processed source into Kaggle working storage before benchmark preparation.


In [ ]:
import os
import subprocess
import sys

REPO_URL = 'https://github.com/hit1363/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System.git'
BRANCH = 'ut'
REPO_DIR = '/kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System'
COPY_DATASET_TO_WORKING = False
WORKING_DATASET_ROOT = '/kaggle/working/leaf_dataset/source'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, '--single-branch', REPO_URL, REPO_DIR], check=True)
elif not os.path.isfile(os.path.join(REPO_DIR, 'training', 'compare_models.py')):
    raise RuntimeError('Existing repository checkout is not the updated ut pipeline; restart the Kaggle session and rerun.')
print('Repository:', REPO_DIR, 'branch:', BRANCH)
print("Copy source dataset to Kaggle working storage:", COPY_DATASET_TO_WORKING)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', os.path.join(REPO_DIR, 'requirements.txt')], check=True)
print('Dependencies installed')

In [ ]:
from pathlib import Path
import shutil
sys.path.insert(0, REPO_DIR)
from dataset.taxonomy import CANONICAL_CLASSES, canonicalize_label

def has_expected_labels(folder):
    if not folder.is_dir():
        return False
    labels = [p.name for p in folder.iterdir() if p.is_dir()]
    mapped = set()
    try:
        mapped = {canonicalize_label(label) for label in labels}
    except ValueError:
        return False
    return len(mapped) == 72

def is_valid_source_root(folder):
    folder = Path(folder)
    split_dirs = [folder / name for name in ("train", "val", "test")]
    present_splits = [path.is_dir() for path in split_dirs]
    if any(present_splits):
        return all(present_splits) and all(has_expected_labels(path) for path in split_dirs)
    return has_expected_labels(folder)

def _kaggle_mount_roots(input_root):
    input_root = Path(input_root)
    if not input_root.is_dir():
        raise FileNotFoundError(f"Kaggle input directory is missing: {input_root}")

    roots = []
    for item in sorted(input_root.iterdir()):
        if not item.is_dir():
            continue
        if item.name.lower() == "datasets":
            for owner in sorted(item.iterdir()):
                if owner.is_dir():
                    roots.extend(child for child in sorted(owner.iterdir()) if child.is_dir())
        else:
            roots.append(item)
    return roots

def find_dataset_source(input_root="/kaggle/input"):
    relative_candidates = (
        ("dataset", "raw"),
        ("raw",),
        ("data", "raw"),
        ("dataset", "processed"),
        ("processed",),
        ("data", "processed"),
        ("dataset",),
        ("data",),
        (),
    )
    seen = set()
    for mount in _kaggle_mount_roots(input_root):
        for parts in relative_candidates:
            candidate = mount.joinpath(*parts)
            key = str(candidate.resolve())
            if key in seen:
                continue
            seen.add(key)
            if is_valid_source_root(candidate):
                return candidate
    raise FileNotFoundError(
        "Could not find a raw, processed, split, or flat source root containing "
        f"the canonical 72 classes under {input_root}."
    )

SOURCE_DIR = find_dataset_source()
if COPY_DATASET_TO_WORKING:
    working_dataset_root = Path(WORKING_DATASET_ROOT)
    if working_dataset_root.exists():
        if not is_valid_source_root(working_dataset_root):
            raise ValueError(
                f"Existing dataset copy is invalid or incomplete: {working_dataset_root}"
            )
        print('Reusing valid source copy:', working_dataset_root)
    else:
        working_dataset_root.parent.mkdir(parents=True, exist_ok=True)
        print('Copying source dataset to Kaggle working storage...')
        shutil.copytree(SOURCE_DIR, working_dataset_root)
        if not is_valid_source_root(working_dataset_root):
            raise ValueError(
                f"Copied dataset failed canonical source validation: {working_dataset_root}"
            )
        print('Source copy complete:', working_dataset_root)
    SOURCE_DIR = working_dataset_root
BENCHMARK_DIR = Path('/kaggle/working/harmonized_benchmark')
print('Source:', SOURCE_DIR)
print('Canonical classes:', len(CANONICAL_CLASSES))

In [ ]:
if BENCHMARK_DIR.exists():
    raise FileExistsError(f'Benchmark output already exists: {BENCHMARK_DIR}. Restart the session or change BENCHMARK_DIR.')
subprocess.run([sys.executable, os.path.join(REPO_DIR, 'dataset', 'prepare_benchmark.py'), '--data-dir', str(SOURCE_DIR), '--output-dir', str(BENCHMARK_DIR), '--seed', '42'], check=True)
print('Prepared benchmark at', BENCHMARK_DIR)

In [ ]:
COMPARISON_DIR = Path('/kaggle/working/leaf_model_comparison')
if COMPARISON_DIR.exists():
    raise FileExistsError(f'Comparison output already exists: {COMPARISON_DIR}. Restart the session or set a new output path.')
subprocess.run([sys.executable, os.path.join(REPO_DIR, 'training', 'compare_models.py'), '--data-dir', str(BENCHMARK_DIR), '--output-dir', str(COMPARISON_DIR)], check=True)
print('Comparison report:', COMPARISON_DIR / 'model_comparison.json')

In [ ]:
import json
import pandas as pd

report = json.loads((COMPARISON_DIR / 'model_comparison.json').read_text())
display(pd.read_csv(COMPARISON_DIR / 'model_comparison.csv').sort_values(['int8_macro_f1', 'int8_mean_inference_ms'], ascending=[False, True]))
print('Pareto frontier:', report['pareto_frontier'])
print(report['latency_note'])